<a href="https://colab.research.google.com/github/viktor-tkachenko/data-analytics-portfolio/blob/main/ab-testing/python/Ecommerce_A_B_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Опис проєкту й посилання

### A/B Testing: Statistical Significance

Розрахунок статистичної значущості для чотирьох конверсійних метрик (add_payment_info, add_shipping_info, begin_checkout, new_accounts на сесію) у чотирьох A/B-тестах мовою Python та візуалізація результатів у Tableau.

**Посилання**
- Дашборд: [A/B Testing: Statistical Significance](https://public.tableau.com/views/ABTestingStatisticalSignificance/Significance?:language=en-GB&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)
- Результати розрахунку (CSV): [ab_test_significance_results.csv](https://drive.google.com/file/d/134Wo0Y5T1a9q1LfsSQHgb6okWPqH7iqC/view?usp=sharing)

**Етапи**
1. Завантаження та очистка даних
2. Конфігурація метрик (статичне місце: назви метрик і спосіб розрахунку)
3. Функція розрахунку значущості (двосторонній z-тест двох пропорцій)
4. Розрахунок у циклі по тестах, метриках і зрізах (тоталь, країна, пристрій, канал, континент)
5. Перевірка результатів: збіг із еталонним дашбордом, позначка `valid_test`
6. Експорт результатів і візуалізація в Tableau

**Підсумок (тоталь)**
- Тест 1: значуще зростання add_payment_info, add_shipping_info і begin_checkout.
- Тест 2: значущих змін немає.
- Тест 3: значуще зниження begin_checkout.
- Тест 4: значуще зниження begin_checkout і new_accounts.

# 2. Імпорти

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import norm

# 3. Підключення до Google Drive та завантаження даних

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/mate_edu
df = pd.read_csv("a_b_testing.csv")

Mounted at /content/drive
/content/drive/MyDrive/mate_edu


In [ ]:
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Slovakia,mobile,Europe,Paid Search,2,2,new_account,1
1,2020-11-04,Bolivia,desktop,Americas,Direct,2,1,new_account,1
2,2020-11-05,Lebanon,mobile,Asia,Direct,2,2,new_account,1
3,2020-11-06,Oman,desktop,Asia,Paid Search,2,2,new_account,1
4,2020-11-06,Dominican Republic,desktop,Americas,Paid Search,2,1,new_account,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB


In [ ]:
display(df['event_name'].unique())
display(df['test'].unique())
display(df['test_group'].unique())

array(['new_account', 'session with orders', 'session', 'page_view',
       'user_engagement', 'scroll', 'view_item', 'session_start',
       'view_promotion', 'click', 'first_visit', 'begin_checkout',
       'add_to_cart', 'select_promotion', 'select_item',
       'add_shipping_info', 'view_search_results', 'add_payment_info',
       'view_item_list'], dtype=object)

array([2, 1, 4, 3])

array([2, 1])

# 4. Очистка й підготовка даних

In [ ]:
# Приведення колонки 'date' до коректного типу даних
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

In [ ]:
# Перевірка відсутності дубльованих рядків
df.duplicated().sum()

np.int64(0)

In [ ]:
# Дублікати за ключем (усі колонки, крім value)
key_cols = ['date', 'country', 'device', 'continent', 'channel', 'test', 'test_group', 'event_name']
df.duplicated(subset=key_cols).sum()

np.int64(0)

In [ ]:
# Знаменник: чи є рядок session для кожної комбінації зрізів
slice_cols = ['date', 'country', 'device', 'continent', 'channel', 'test', 'test_group']
all_keys = df[slice_cols].drop_duplicates()
sess_keys = df.loc[df['event_name'] == 'session', slice_cols].drop_duplicates()

check_keys = all_keys.merge(sess_keys, on=slice_cols, how='left', indicator=True)
print('Комбінацій зрізів:', len(all_keys),
      '| із рядком session:', len(sess_keys),
      '| без session:', (check_keys['_merge'] == 'left_only').sum())

Комбінацій зрізів: 107210 | із рядком session: 107210 | без session: 0


In [ ]:
# Баланс груп за кількістю сесій у кожному тесті
balance = (df[df['event_name'] == 'session']
           .pivot_table(index='test', columns='test_group', values='value', aggfunc='sum'))
balance['group_2_%'] = (balance[2] / (balance[1] + balance[2]) * 100).round(2)
display(balance)

test_group,1,2,group_2_%
test,,,
1,45362,45193,49.91
2,50637,50244,49.81
3,70047,70439,50.14
4,105079,105141,50.01


In [ ]:
# Перевірка відсутності пропущених значень
df.isna().sum()

,0
date,0
country,0
device,0
continent,0
channel,0
test,0
test_group,0
event_name,0
value,0


# 5. Конфігурація метрик

Розрахунок значущості не залежить від кількості й складу метрик: усе, що задається статично, зібрано в цьому розділі. Метрика описується парою подій у даних: чисельник (подія, яку рахуємо) і знаменник (сесії). Щоб додати нову метрику, достатньо додати один запис у конфігурацію; логіка розрахунку (розділи 6-7) не змінюється.

Окремо задаються параметри тесту (рівень значущості, номери контрольної й тестової груп) і список зрізів, у яких проводиться розрахунок.

In [ ]:
# 1. Метрики: назва, подія-чисельник, подія-знаменник.
METRICS = [
    {'metric': 'add_payment_info / session',  'numerator_event': 'add_payment_info',  'denominator_event': 'session'},
    {'metric': 'add_shipping_info / session', 'numerator_event': 'add_shipping_info', 'denominator_event': 'session'},
    {'metric': 'begin_checkout / session',    'numerator_event': 'begin_checkout',    'denominator_event': 'session'},
    {'metric': 'new_accounts / session',      'numerator_event': 'new_account',       'denominator_event': 'session'},
]

# 2. Зрізи: кожна колонка розглядається окремо
SLICES = ['country', 'device', 'channel', 'continent']

# 3. Параметри тесту
ALPHA = 0.05
CONTROL_GROUP = 1
TEST_GROUP = 2

# 4. Події, потрібні для розрахунку: береться з конфігу
needed_events = sorted({m[key] for m in METRICS for key in ('numerator_event', 'denominator_event')})

# 5. Перевірка: усі події з конфігу є в даних
missing_events = set(needed_events) - set(df['event_name'].unique())
print('Потрібні події:', needed_events)
print('Відсутні в даних:', missing_events)

Потрібні події: ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_account', 'session']
Відсутні в даних: set()


# 6. Функція розрахунку статистичної значущості

Для кожної метрики порівнюємо частку сесій із подією в контрольній (1) і тестовій (2) групах z-тестом двох пропорцій.

- **Нульова гіпотеза:** справжні конверсії в групах однакові, різниця зумовлена випадковістю.
- **Конверсії:** p₁ = x₁ / n₁ (контроль), p₂ = x₂ / n₂ (тест), де x - це кількість подій, n - це кількість сесій.
- **Зміна метрики:** (p₂ − p₁) / p₁ × 100, тестова група відносно контрольної.
- **Спільна частка:** p = (x₁ + x₂) / (n₁ + n₂).
- **Стандартна помилка:** SE = √( p(1 − p)(1/n₁ + 1/n₂) ).
- **z-статистика:** z = (p₂ − p₁) / SE.
- **p-value:** двостороннє, 2·(1 − Φ(|z|)), де Φ це функція стандартного нормального розподілу.
- **Значущість:** результат значущий, якщо p-value менше за рівень значущості (0,05).

Якщо в групі немає сесій або подій немає в обох групах, статистика не визначена: повертаємо порожні значення, а результат вважаємо незначущим.

In [ ]:
def significance_test(x_control, n_control, x_test, n_test, alpha=ALPHA):
    """Z-тест двох пропорцій: тестова група відносно контрольної."""
    result = {
        'control_conversion': np.nan,
        'test_conversion': np.nan,
        'metric_change_%': np.nan,
        'z_stat': np.nan,
        'p_value': np.nan,
        'significant': False,
    }

    # Немає сесій хоча б в одній групі: тест неможливий
    if n_control == 0 or n_test == 0:
        return result

    # Конверсії та зміна
    p_control = x_control / n_control
    p_test = x_test / n_test
    result['control_conversion'] = p_control
    result['test_conversion'] = p_test
    if p_control > 0:
        result['metric_change_%'] = (p_test - p_control) / p_control * 100

    # Спільна частка та стандартна помилка
    p_pooled = (x_control + x_test) / (n_control + n_test)
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_control + 1 / n_test))
    if se == 0:   # подій немає в обох групах (або вони в усіх сесіях)
        return result

    # z-статистика та двостороннє p-value
    z_stat = (p_test - p_control) / se
    p_value = 2 * norm.sf(abs(z_stat))   # те саме, що 2 * (1 - norm.cdf(|z|)), але точніше для малих значень

    result['z_stat'] = z_stat
    result['p_value'] = p_value
    result['significant'] = bool(p_value < alpha)
    return result

# Перевірка на прикладі зі старого дашборда (тест 4, add_payment_info, всього)
print(significance_test(3731, 105079, 3601, 105141))
# Граничні випадки
print(significance_test(0, 0, 0, 0))        # немає сесій
print(significance_test(0, 100, 0, 100))    # подій немає в обох групах

{'control_conversion': 0.03550661882964246, 'test_conversion': 0.034249246250273445, 'metric_change_%': -3.541234341128839, 'z_stat': np.float64(-1.5711064424503227), 'p_value': np.float64(0.11615792640849863), 'significant': False}
{'control_conversion': nan, 'test_conversion': nan, 'metric_change_%': nan, 'z_stat': nan, 'p_value': nan, 'significant': False}
{'control_conversion': 0.0, 'test_conversion': 0.0, 'metric_change_%': nan, 'z_stat': nan, 'p_value': nan, 'significant': False}


# 7. Розрахунок значущості для всіх метрик і зрізів

## 7.1. Підготовка лічильників

Вхідна таблиця містить події по днях. Для тесту дні не потрібні, тому складаємо `value` за весь період окремо для кожної комбінації: тест, група, зріз, подія. Зрізи беремо по одному (тотал, країна, пристрій, канал, континент), без комбінацій між ними. Потрібні події визначаються з конфігурації метрик (розділ 5), а не вписуються вручну.

Далі події розгортаємо в колонки: один рядок на комбінацію «тест, зріз, група», а події стоять окремими колонками. Якщо події в зрізі не було, рядка для неї в даних немає, тому відсутні значення замінюємо нулем.

In [ ]:
# 1. Лише потрібні події (список береться з конфігу)
base = df[df['event_name'].isin(needed_events)]

frames = []

# 2. Тотал: без розрізу
total = base.groupby(['test', 'test_group', 'event_name'], as_index=False)['value'].sum()
total['slice_type'] = 'total'
total['slice_value'] = 'all'
frames.append(total)

# 3. Кожен зріз окремо
for slice_col in SLICES:
    part = (base.groupby(['test', 'test_group', slice_col, 'event_name'], as_index=False)['value'].sum()
                .rename(columns={slice_col: 'slice_value'}))
    part['slice_type'] = slice_col
    frames.append(part)

counts_long = pd.concat(frames, ignore_index=True)

# 4. Події в колонки, відсутні значення = 0
counts = counts_long.pivot_table(
    index=['test', 'slice_type', 'slice_value', 'test_group'],
    columns='event_name',
    values='value',
    aggfunc='sum',
    fill_value=0
)

# 5. Перевірки
print('Рядків у counts:', len(counts))
print('Тести:', sorted(counts.index.get_level_values('test').unique()))
print('Зрізи:', counts.index.get_level_values('slice_type').unique().tolist())
display(counts.loc[(4, 'total', 'all')])   # тест 4, тотал: має збігтись зі старим дашбордом

Рядків у counts: 984
Тести: [1, 2, 3, 4]
Зрізи: ['channel', 'continent', 'country', 'device', 'total']


event_name,add_payment_info,add_shipping_info,begin_checkout,new_account,session
test_group,,,,,
1,3731,5128,12555,8984,105079
2,3601,4956,12267,8687,105141


## 7.2. Цикл по тестах, метриках і зрізах

Для кожної комбінації «тест, тип зрізу, значення зрізу» беремо з таблиці лічильників два рядки (контрольна й тестова група), а для кожної метрики з конфігурації дістаємо чотири числа: подію й сесії обох груп. Їх передаємо у функцію з розділу 6 і складаємо результати в таблицю. Якщо в зрізі немає однієї з груп, її лічильники вважаємо нулями: тест у такому випадку неможливий, і функція повертає порожні значення.

In [ ]:
results = []

# 1. Зовнішній цикл: кожна комбінація тест + зріз + значення зрізу
for (test, slice_type, slice_value), g in counts.groupby(level=['test', 'slice_type', 'slice_value']):
    g = g.droplevel(['test', 'slice_type', 'slice_value'])      # лишається індекс test_group
    g = g.reindex([CONTROL_GROUP, TEST_GROUP], fill_value=0)    # відсутня група = нулі

    # 2. Внутрішній цикл: кожна метрика з конфігу
    for m in METRICS:
        num, den = m['numerator_event'], m['denominator_event']

        # 3. Чотири числа для функції
        x_c, n_c = int(g.loc[CONTROL_GROUP, num]), int(g.loc[CONTROL_GROUP, den])
        x_t, n_t = int(g.loc[TEST_GROUP, num]), int(g.loc[TEST_GROUP, den])

        # 4. Виклик функції з розділу 6
        res = significance_test(x_c, n_c, x_t, n_t)

        # 5. Рядок результату
        results.append({
            'test_number': test,
            'metric': m['metric'],
            'numerator_event': num,
            'denominator_event': den,
            'slice_type': slice_type,
            'slice_value': slice_value,
            'control_numerator_count': x_c,
            'control_denominator_count': n_c,
            'control_conversion_rate': res['control_conversion'],
            'test_numerator_count': x_t,
            'test_denominator_count': n_t,
            'test_conversion_rate': res['test_conversion'],
            'metric_change_%': res['metric_change_%'],
            'z_stat': res['z_stat'],
            'p_value': res['p_value'],
            'significant': res['significant'],
        })

results_df = pd.DataFrame(results)

# 6. Перевірки
n_combos = counts.reset_index()[['test', 'slice_type', 'slice_value']].drop_duplicates().shape[0]
print('Рядків у результаті:', len(results_df), '| очікувано:', n_combos * len(METRICS))
display(results_df[(results_df['test_number'] == 4) & (results_df['slice_type'] == 'total')])

Рядків у результаті: 1968 | очікувано: 1968


/tmp/ipykernel_1966/1596735303.py:26: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_control + 1 / n_test))


,test_number,metric,numerator_event,denominator_event,slice_type,slice_value,control_numerator_count,control_denominator_count,control_conversion_rate,test_numerator_count,test_denominator_count,test_conversion_rate,metric_change_%,z_stat,p_value,significant
1964,4,add_payment_info / session,add_payment_info,session,total,all,3731,105079,0.035507,3601,105141,0.034249,-3.541234,-1.571106,0.116158,False
1965,4,add_shipping_info / session,add_shipping_info,session,total,all,5128,105079,0.048801,4956,105141,0.047137,-3.411125,-1.785795,0.074132,False
1966,4,begin_checkout / session,begin_checkout,session,total,all,12555,105079,0.119482,12267,105141,0.116672,-2.351523,-1.995998,0.045934,True
1967,4,new_accounts / session,new_account,session,total,all,8984,105079,0.085498,8687,105141,0.082622,-3.362896,-2.375457,0.017527,True


# 8. Перевірка результатів

Перед експортом перевіряємо розрахунок:
- структуру таблиці (кількість рядків, порожні ключові поля, кількість метрик);
- збіг тесту 4 (тотал) зі значеннями еталонного дашборда;
- придатність кожного тесту. Якщо подій у зрізі більше, ніж сесій (запит рахує події, а не сесії з подією), конверсія виходить понад 100%, і статистику розрахувати неможливо. Такі рядки не видаляємо, а позначаємо колонкою `valid_test`.

In [ ]:
# 1. Структура таблиці
print('Рядків:', len(results_df))
print('Порожніх у ключових полях:',
      results_df[['test_number', 'metric', 'slice_type', 'slice_value']].isna().sum().sum())
print('Метрик у результаті:', results_df['metric'].nunique(), '| у конфігу:', len(METRICS))

# 2. Збіг тесту 4 (тоталь) з еталонним дашбордом: зміна, p-value, значущість
reference = {
    'add_payment_info / session':  (-3.541, 0.116, False),
    'add_shipping_info / session': (-3.411, 0.074, False),
    'begin_checkout / session':    (-2.352, 0.046, True),
    'new_accounts / session':      (-3.363, 0.018, True),
}
check = results_df[(results_df['test_number'] == 4) & (results_df['slice_type'] == 'total')].set_index('metric')
for metric, (change, p, sig) in reference.items():
    row = check.loc[metric]
    ok = (np.isclose(row['metric_change_%'], change, atol=0.001)
          and np.isclose(row['p_value'], p, atol=0.001)
          and row['significant'] == sig)
    print(f'{metric}: {"збігається" if ok else "НЕ збігається"}')

# 3. Придатність тесту
over = (results_df['control_conversion_rate'] > 1) | (results_df['test_conversion_rate'] > 1)
results_df['valid_test'] = (
    (results_df['control_denominator_count'] > 0)
    & (results_df['test_denominator_count'] > 0)
    & ~over
    & results_df['z_stat'].notna()
)
print('\nРядків із конверсією понад 100%:', over.sum())
print(results_df['valid_test'].value_counts())
print(results_df.groupby('slice_type')['valid_test'].mean().round(3))

# 4. Де саме непридатні тести
bad = results_df[~results_df['valid_test']]
display(bad.groupby(['slice_type', 'metric']).size().unstack(fill_value=0))
display(bad[['test_number', 'metric', 'slice_type', 'slice_value',
             'control_numerator_count', 'control_denominator_count',
             'test_numerator_count', 'test_denominator_count']].head(10))

Рядків: 1968
Порожніх у ключових полях: 0
Метрик у результаті: 4 | у конфігу: 4
add_payment_info / session: збігається
add_shipping_info / session: збігається
begin_checkout / session: збігається
new_accounts / session: збігається

Рядків із конверсією понад 100%: 1
valid_test
True     1810
False     158
Name: count, dtype: int64
slice_type
channel      1.000
continent    1.000
country      0.909
device       1.000
total        1.000
Name: valid_test, dtype: float64


metric,add_payment_info / session,add_shipping_info / session,begin_checkout / session,new_accounts / session
slice_type,,,,
country,68,41,42,7


,test_number,metric,slice_type,slice_value,control_numerator_count,control_denominator_count,test_numerator_count,test_denominator_count
60,1,add_payment_info / session,country,Armenia,0,8,0,11
76,1,add_payment_info / session,country,Bahamas,0,6,0,6
77,1,add_shipping_info / session,country,Bahamas,0,6,0,6
78,1,begin_checkout / session,country,Bahamas,0,6,0,6
83,1,new_accounts / session,country,Bahrain,0,6,0,8
88,1,add_payment_info / session,country,Belarus,0,17,0,13
96,1,add_payment_info / session,country,Bolivia,0,4,0,16
97,1,add_shipping_info / session,country,Bolivia,0,4,0,16
98,1,begin_checkout / session,country,Bolivia,0,4,0,16
100,1,add_payment_info / session,country,Bosnia & Herzegovina,0,13,0,7


**Висновок за перевіркою.**
- Структура результату коректна: 1 968 рядків (4 тести × 4 метрики × 123 значення зрізів), жодного порожнього ключового поля, усі метрики з конфігурації присутні.
- Тест 4 (тоталь) збігається з еталонним дашбордом за всіма чотирма метриками (зміна, p-value, значущість).
- У 158 рядках (8%) статистику розрахувати неможливо; усі вони належать до зрізу за країнами. Переважно це країни з малою кількістю сесій, де подій немає в обох групах (шум дорівнює нулю, z не визначений); в одному рядку подій більше, ніж сесій (запит рахує події, а не сесії з подією), тому конверсія перевищує 100%.
- Такі рядки не видаляємо, а позначаємо колонкою `valid_test = False`: порожній результат означає «замало даних», а не «різниці немає». У дашборді їх можна відфільтрувати.

# 9. Експорт результатів

Результат зберігаємо в один CSV-файл, який є джерелом даних для дашборда в Tableau. Один рядок описує один тест однієї метрики в одному зрізі (тоталь, країна, пристрій, канал або континент). Порядок колонок відповідає прикладу з завдання; контрольна група (1) і тестова (2) позначені префіксами `control_` і `test_`, зміна метрики рахується як тестова група відносно контрольної. Колонка `valid_test` показує, чи вдалося розрахувати статистику.

In [ ]:
# 1. Порядок колонок і сортування
export_cols = [
    'test_number', 'metric', 'numerator_event', 'denominator_event',
    'slice_type', 'slice_value',
    'control_numerator_count', 'control_denominator_count', 'control_conversion_rate',
    'test_numerator_count', 'test_denominator_count', 'test_conversion_rate',
    'metric_change_%', 'z_stat', 'p_value', 'significant', 'valid_test'
]
export_df = (results_df[export_cols]
             .sort_values(['test_number', 'metric', 'slice_type', 'slice_value'])
             .reset_index(drop=True))

# Допоміжні колонки для дашборда
export_df['significance_label'] = np.where(~export_df['valid_test'], 'not enough data',
                                  np.where(export_df['significant'], 'significant', 'not significant'))
export_df['significant_direction'] = np.where(~export_df['valid_test'] | ~export_df['significant'], 'none',
                                     np.where(export_df['metric_change_%'] > 0, 'positive', 'negative'))
export_df.to_csv('ab_test_significance_results.csv', index=False, encoding='utf-8')
print(export_df['significance_label'].value_counts())

# 2. Збереження на Google Drive (у ту саму папку, що й вхідний файл)
# export_df.to_csv('ab_test_significance_results.csv', index=False, encoding='utf-8')

# 3. Перевірка: читаємо збережений файл назад
check_df = pd.read_csv('ab_test_significance_results.csv')
print('Рядків:', len(check_df), '| колонок:', check_df.shape[1])
print('Збігається з результатом у пам\'яті:', len(check_df) == len(export_df))
display(check_df.head())

significance_label
not significant    1434
significant         376
not enough data     158
Name: count, dtype: int64
Рядків: 1968 | колонок: 19
Збігається з результатом у пам'яті: True


,test_number,metric,numerator_event,denominator_event,slice_type,slice_value,control_numerator_count,control_denominator_count,control_conversion_rate,test_numerator_count,test_denominator_count,test_conversion_rate,metric_change_%,z_stat,p_value,significant,valid_test,significance_label,significant_direction
0,1,add_payment_info / session,add_payment_info,session,channel,Direct,392,10691,0.036666,516,10361,0.049802,35.825180,4.690261,2.728566e-06,True,True,significant,positive
1,1,add_payment_info / session,add_payment_info,session,channel,Organic Search,640,15675,0.040829,514,15631,0.032883,-19.461427,-3.730758,1.909045e-04,True,True,significant,negative
2,1,add_payment_info / session,add_payment_info,session,channel,Paid Search,450,11777,0.038210,519,11841,0.043831,14.709963,2.177389,2.945153e-02,True,True,significant,positive
3,1,add_payment_info / session,add_payment_info,session,channel,Social Search,237,3883,0.061035,264,3963,0.066616,9.143757,1.010906,3.120614e-01,False,True,not significant,none
4,1,add_payment_info / session,add_payment_info,session,channel,Undefined,269,3336,0.080635,416,3397,0.122461,51.869844,5.676166,1.377470e-08,True,True,significant,positive


# 10. Візуалізація результатів у Tableau

Результати розрахунку (CSV із розділу 9) візуалізовано в оновленому дашборді A/B-тестування: [Дашборд у Tableau Public](https://public.tableau.com/views/ABTestingStatisticalSignificance/Significance?:language=en-GB&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link).

- **Значущість:** таблиця для вибраного тесту показує чотири метрики: конверсії контрольної та тестової груп, зміну метрики, z-статистику, p-value; значущі зміни виділено кольором (напрям: позитивний чи негативний), рядки з недостатньою кількістю даних позначено окремо.
- **Фільтри:** за номером тесту, типом і значенням зрізу (тоталь, країна, пристрій, канал, континент).
- **Пояснення логіки:** на дашборді наведено опис тесту (z-тест двох пропорцій, H0, рівень значущості 0,05, формули) і посилання на цей ноутбук та CSV.

# 11. Висновки

**Результати в тоталі по тестах** (група 2 відносно групи 1, двосторонній z-тест двох пропорцій, α = 0,05):

| Тест | add_payment_info | add_shipping_info | begin_checkout | new_accounts |
|---|---|---|---|---|
| 1 | +12,54%, p = 0,0001 | +6,56%, p = 0,0092 | +6,66%, p = 0,0029 | −3,35%, p = 0,123 |
| 2 | +3,58%, p = 0,215 | +1,65%, p = 0,478 | +1,99%, p = 0,341 | +1,24%, p = 0,556 |
| 3 | +1,48%, p = 0,520 | −2,62%, p = 0,157 | −3,35%, p = 0,012 | −1,13%, p = 0,520 |
| 4 | −3,54%, p = 0,116 | −3,41%, p = 0,074 | −2,35%, p = 0,046 | −3,36%, p = 0,018 |

- **Тест 1:** значуще зростання трьох метрик воронки (add_payment_info, add_shipping_info, begin_checkout); new_accounts значущої зміни не показує.
- **Тест 2:** жодна метрика не змінилась значуще.
- **Тест 3:** значуще лише зниження begin_checkout.
- **Тест 4:** значуще зниження begin_checkout і new_accounts; для add_payment_info і add_shipping_info значущої зміни немає.

**Розрахунок у розрізах.** Окрім тоталі, тести проведено окремо для кожної країни, пристрою, каналу та континенту (1 968 рядків результату). Рядки, де статистику розрахувати неможливо (подій немає в обох групах або їх більше, ніж сесій), позначено `valid_test = False` і виключено з візуалізації значущості.
